In [0]:
# Configuration
from pyspark.sql import functions as F, Window
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
dbutils.widgets.combobox("gold_schema", "gold", ["gold", "gabrielajaniszews786_gold"], "Gold schema")
CATALOG       = dbutils.widgets.get("catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")


In [0]:
# Creating a dim table with different date grains
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.dim_date (
  date                  DATE     NOT NULL,
  month                 INTEGER     NOT NULL,
  day                   INTEGER     NOT NULL,
  week                  INTEGER     NOT NULL,
  year                  INTEGER     NOT NULL,
  day_of_week           INTEGER  NOT NULL,
  day_of_week_name      STRING   NOT NULL,
  week_of_year          INTEGER  NOT NULL,
  month_name            STRING    NOT NULL,
  quarter               INTEGER  NOT NULL,
  is_weekend            STRING  NOT NULL,
  CONSTRAINT pk_dim_date PRIMARY KEY (date))
USING DELTA
""")

In [0]:
# Populating dim_date table with date grains from 2026-06-01 to 2026-12-31
spark.sql(f"""
WITH RECURSIVE Dates
MAX RECURSION LEVEL 3000
AS
(
 SELECT 
 CAST('2026-06-01' as date) as date,
 month(cast('2026-06-01' as date)) as month,
 dayofmonth(cast('2026-06-01' as date)) as day,
 weekofyear(cast('2026-06-01' as date)) as week,
 year(cast('2026-06-01' as date)) as year,
 dayofweek(cast('2026-06-01' as date)) as day_of_week,
 dayname(cast('2026-06-01' as date)) as day_of_week_name,
 weekofyear(cast('2026-06-01' as date)) as week_of_year,
 date_format(cast('2026-06-01' as date), 'MMMM') as month_name,
 quarter(cast('2026-06-01' as date)) as quarter,
 CASE WHEN dayofweek(cast('2026-06-01' as date)) IN (1, 7) THEN 'Yes' ELSE 'No' END as is_weekend

 UNION ALL
 
 SELECT
 CAST(dateadd(day, 1, date) as date) AS date,
 month(dateadd(day, 1, date)) as month,
 dayofmonth(dateadd(day, 1, date)) as day,
 weekofyear(dateadd(day, 1, date)) as week,
 year(dateadd(day, 1, date)) as year,
 dayofweek(dateadd(day, 1, date)) as day_of_week,
 dayname(dateadd(day, 1, date)) as day_of_week_name,
 weekofyear(dateadd(day, 1, date)) as week_of_year,
 date_format(dateadd(day, 1, date), 'MMMM') as month_name,
 quarter(dateadd(day, 1, date)) as quarter,
 CASE WHEN dayofweek(dateadd(day, 1, date)) IN (1, 7) THEN 'Yes' ELSE 'No' END as is_weekend
 FROM Dates
 WHERE dateadd(day, 1, date) <= cast('2026-12-31' as date)
)
INSERT OVERWRITE {CATALOG}.{GOLD_SCHEMA}.dim_date
SELECT *
FROM Dates
""")

display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.dim_date LIMIT 5"))
